# Exploration & Testing Notebook
Apartment Rental Agreement Red Flag Agent -- prototyping and sanity checks for the pipeline stages.
Run from the project root, or adjust the path insert below.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src import ingestion, preprocessing, classifier, risk_scoring, explainer, negotiation_coach
from src.main import process_agreement

## 1. Load and clean a sample agreement

In [ ]:
raw = ingestion.load_text('../data/sample_agreement_2.txt')
cleaned = preprocessing.clean_text(raw)
print(cleaned[:500])

## 2. Segment into clauses

In [ ]:
clauses = preprocessing.segment_clauses(cleaned)
for c in clauses:
    print(c.clause_id, '->', c.original_text[:60].replace(chr(10), ' '))

## 3. Classify each clause

In [ ]:
clauses = classifier.classify_clauses(clauses)
for c in clauses:
    print(c.clause_id, c.category)

## 4. Score risk for each clause

In [ ]:
for c in clauses:
    score = risk_scoring.score_clause(c)
    print(c.clause_id, c.category, '->', score['risk_level'], '(', round(score['confidence'],2), ')', '-', score['reason'])

## 5. Run the full pipeline end to end

In [ ]:
result = process_agreement(file_path='../data/sample_agreement_3.txt', log_result=False)
for f in result['flags']:
    print(f['clause_id'], f['category'], f['risk_level'])
    print('  ->', f['suggested_question_to_landlord'])

## 6. Compare all three sample agreements at a glance

In [ ]:
for path in ['../data/sample_agreement_1.txt', '../data/sample_agreement_2.txt', '../data/sample_agreement_3.txt']:
    r = process_agreement(file_path=path, log_result=False)
    red = sum(1 for f in r['flags'] if f['risk_level']=='red')
    yellow = sum(1 for f in r['flags'] if f['risk_level']=='yellow')
    green = sum(1 for f in r['flags'] if f['risk_level']=='green')
    print(path, '-> red:', red, 'yellow:', yellow, 'green:', green)